# hgt-site quickstart

This notebook runs hgt-site end to end in about a minute. It needs no data of your own and nothing is downloaded: hgt-site can write its own small dataset with known insertion sites, so everything the tool reports can be checked against the right answer.

hgt-site looks for places where DNA from bacteria or other organisms may have been inserted into a host genome. The hard part is that most bacterial reads in a host sample are contamination, or fragments joined by chance during library preparation, so read support on its own is not evidence of an insertion. hgt-site tests each candidate against the number of pairs chance alone would put there.

Code and benchmarks: https://github.com/iraajgangavaram/hgt-site  
Documentation: https://iraajgangavaram.github.io/hgt-site/

## Install

In [ ]:
%pip install -q hgt-site

# For the development version instead:
# %pip install -q "git+https://github.com/iraajgangavaram/hgt-site"

!hgt-site --version

## Make a dataset with known answers

`hgt-site simulate` writes four files: an alignment of paired-end reads against a small host genome, Kraken2 classifications for the reads that did not align, a BED file listing the insertions it put in, and the host sequence. The dataset contains real insertions, contaminating bacterial reads and chimeric pairs that join host and bacterial DNA at arbitrary positions, which is what makes the problem hard.

In [ ]:
!hgt-site simulate -o demo --seed 1

print("\nThe insertions that were put in (chrom, start, end, donor taxid, length):")
!cat demo.truth.bed

## Call sites

`hgt-site call` takes the alignment and the Kraken2 output. `--host-taxid 9606` tells it which taxon is the host, so host reads are not treated as foreign.

The messages it prints are worth reading. They say how many read pairs had one mate on the host, how many of those had a Kraken2 call from outside the host, and what fragment size it measured from the properly paired reads, which it uses to set its clustering limits.

In [ ]:
!hgt-site call \
    --bam demo.sam \
    --kraken demo.kraken2.txt \
    --host-taxid 9606 \
    -o sites.tsv \
    --bed sites.bed

## Check the answers

The table holds every candidate site, with a `filter` column saying whether it passed or which test it failed. Below, each reported site is compared with the insertions the simulator actually put in, and the same comparison is made for the obvious alternative: keeping every candidate with enough supporting read pairs.

In [ ]:
import pandas as pd

sites = pd.read_csv("sites.tsv", sep="\t")
truth = pd.read_csv(
    "demo.truth.bed", sep="\t", header=None,
    names=["chrom", "start", "end", "donor_taxid", "length"],
)


def is_real(site, slop=500):
    """True if a real insertion sits inside this site, give or take slop bases."""
    same_chrom = truth[truth.chrom == site.chrom]
    return bool(((same_chrom.start >= site.start - slop)
                 & (same_chrom.start <= site.end + slop)).any())


passed = sites[sites["filter"] == "PASS"]
support_only = sites[sites["n_pairs"] >= 3]

print(f"candidate sites considered:      {len(sites)}")
print(f"real insertions in the dataset:  {len(truth)}\n")
for label, table in [("hgt-site, after its tests", passed),
                     ("read support alone", support_only)]:
    real = int(table.apply(is_real, axis=1).sum()) if len(table) else 0
    print(f"{label:28} {len(table):>3} sites reported, {real} of them real")

Read support alone finds every real insertion here and adds a few sites that are not there. That gap is small on a dataset this quiet. On the benchmark datasets, which carry realistic numbers of chimeric pairs, the same filter reports thousands of sites that are not insertions, while hgt-site reported none across 18 runs on fresh random seeds. The [benchmarks](https://iraajgangavaram.github.io/hgt-site/benchmarks/) have the measurements and the conditions.

Here are the sites that passed, with the evidence for each one. `expected_pairs` is how many pairs chance alone would have put at that position given how common that taxon is across the whole sample, and `qvalue` is the enrichment test corrected for the number of positions tested.

In [ ]:
passed[["chrom", "start", "end", "n_pairs", "top_taxid",
        "top_taxid_fraction", "expected_pairs", "qvalue"]]

## On your own data

You need paired-end reads aligned to the host genome, keeping the pairs where one mate did not align, and Kraken2 classifications for those unmapped reads. If you have a Kraken2 database to hand, one command does the lot:

```bash
hgt-site run --bam sample.bam --kraken-db /path/to/kraken2-db \
    --host-taxid 9606 --threads 8 -o sample.hgt-site.tsv --vcf sample.hgt-site.vcf
```

If your pipeline already runs Kraken2 on the unmapped reads, pass its output to `hgt-site call --kraken` instead and skip the classification step.

Where reads cross the junction itself and are soft-clipped past it, hgt-site reports the insertion point to the base pair. That needs an indexed BAM or CRAM, so it does not appear in this notebook, which uses a plain SAM file.

Before trusting anything it reports, read the [limitations](https://iraajgangavaram.github.io/hgt-site/benchmarks/#limitations). It has not yet been run on real sequencing data with confirmed integrations, so a candidate site is a lead to follow up, not a finding.

Problems and criticism of the method are welcome in [GitHub issues](https://github.com/iraajgangavaram/hgt-site/issues).